In [ ]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.6 MB/s eta 0:00:00


In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# =========================
# BASE DE DATOS (MEMORIA)
# =========================

pacientes = []
medicos = []
citas = []
consultas = []

# =========================
# LISTAS
# =========================

TIPOS_SANGRE = ["O+", "O-", "A+", "A-", "B+", "B-", "AB+", "AB-"]
REGIMENES = ["CONTRIBUTIVO", "SUBSIDIADO"]

ESPECIALIDADES = [
    "Medicina General", "Pediatría", "Cardiología",
    "Ortopedia", "Dermatología", "Neurología", "Psiquiatría"
]

CONSULTORIOS = ["A1", "A2", "A3", "B1", "B2", "B3"]
HORARIOS = ["08:00-20:00", "20:00-08:00"]

# =========================
# VALIDACIONES
# =========================

def validar_nombre(change):
    if any(char.isdigit() for char in change['new']):
        nombre.value = ''.join(filter(lambda x: not x.isdigit(), change['new']))

def validar_fecha(texto):
    try:
        datetime.strptime(texto, "%Y/%m/%d")
        return True
    except:
        return False

# =========================
# PACIENTES
# =========================

def vista_pacientes():
    clear_output()

    global nombre

    doc = widgets.Text(description="Documento:")
    nombre = widgets.Text(description="Nombre:")
    nombre.observe(validar_nombre, names='value')

    fecha = widgets.Text(description="Nacimiento:")
    sangre = widgets.Dropdown(options=TIPOS_SANGRE, description="Sangre:")
    eps = widgets.Text(description="EPS:")
    regimen = widgets.Dropdown(options=REGIMENES, description="Régimen:")
    ant = widgets.Text(description="Antecedentes:")

    btn = widgets.Button(description="Guardar")

    def guardar(b):
        if not validar_fecha(fecha.value):
            print("❌ Fecha inválida (AAAA/MM/DD)")
            return

        pacientes.append({
            "doc": doc.value,
            "nombre": nombre.value,
            "fecha": fecha.value,
            "sangre": sangre.value,
            "eps": eps.value,
            "regimen": regimen.value,
            "ant": ant.value
        })

        print("✔ Paciente registrado")

    btn.on_click(guardar)

    display(doc, nombre, fecha, sangre, eps, regimen, ant, btn)

    # EDITAR
    if pacientes:
        docs = [p["doc"] for p in pacientes]

        selector = widgets.Dropdown(options=docs, description="Editar:")

        btn_edit = widgets.Button(description="Editar")

        def editar(b):
            p = next(x for x in pacientes if x["doc"] == selector.value)

            nombre.value = p["nombre"]
            fecha.value = p["fecha"]

            def guardar_edit(b):
                p["nombre"] = nombre.value
                p["fecha"] = fecha.value
                print("✔ Actualizado")

            btn.on_click(guardar_edit)

        btn_edit.on_click(editar)

        display(selector, btn_edit)

# =========================
# MÉDICOS
# =========================

def vista_medicos():
    clear_output()

    doc = widgets.Text(description="Registro:")
    nombre = widgets.Text(description="Nombre:")
    nombre.observe(validar_nombre, names='value')

    esp = widgets.Dropdown(options=ESPECIALIDADES, description="Especialidad:")
    cons = widgets.Dropdown(options=CONSULTORIOS, description="Consultorio:")
    horario = widgets.Dropdown(options=HORARIOS, description="Horario:")

    btn = widgets.Button(description="Guardar")

    def guardar(b):
        medicos.append({
            "doc": doc.value,
            "nombre": nombre.value,
            "esp": esp.value,
            "cons": cons.value,
            "horario": horario.value
        })
        print("✔ Médico registrado")

    btn.on_click(guardar)

    display(doc, nombre, esp, cons, horario, btn)

    # FILTROS
    filtro_esp = widgets.Dropdown(options=ESPECIALIDADES, description="Filtrar Esp:")
    btn_filtro = widgets.Button(description="Buscar")

    def buscar(b):
        for m in medicos:
            if m["esp"] == filtro_esp.value:
                print(m)

    btn_filtro.on_click(buscar)

    display(filtro_esp, btn_filtro)

# =========================
# CITAS
# =========================

def generar_horas(horario):
    inicio, fin = horario.split("-")
    horas = []
    h = int(inicio[:2])
    while h != int(fin[:2]):
        horas.append(f"{h:02d}:00")
        horas.append(f"{h:02d}:30")
        h = (h + 1) % 24
    return horas

def vista_citas():
    clear_output()

    if not pacientes or not medicos:
        print("❌ Debes registrar pacientes y médicos")
        return

    doc = widgets.Dropdown(options=[p["doc"] for p in pacientes], description="Paciente:")
    med = widgets.Dropdown(options=[m["doc"] for m in medicos], description="Médico:")

    fecha = widgets.Text(description="Fecha:")

    hora = widgets.Dropdown(options=[], description="Hora:")

    def actualizar_horas(change):
        m = next(x for x in medicos if x["doc"] == med.value)
        hora.options = generar_horas(m["horario"])

    med.observe(actualizar_horas, names='value')

    motivo = widgets.Text(description="Motivo:")
    btn = widgets.Button(description="Crear cita")

    def guardar(b):
        cod = str(len(citas)+1).zfill(3)

        citas.append({
            "cod": cod,
            "doc": doc.value,
            "med": med.value,
            "fecha": fecha.value,
            "hora": hora.value,
            "motivo": motivo.value,
            "estado": "ACTIVA"
        })

        print("✔ Cita creada:", cod)

    btn.on_click(guardar)

    display(doc, med, fecha, hora, motivo, btn)

    # CONSULTAR / CANCELAR
    if citas:
        cods = [c["cod"] for c in citas]

        sel = widgets.Dropdown(options=cods, description="Cita:")
        btn_cons = widgets.Button(description="Consultar")
        btn_can = widgets.Button(description="Cancelar")

        def consultar(b):
            c = next(x for x in citas if x["cod"] == sel.value)
            print(c)

        def cancelar(b):
            c = next(x for x in citas if x["cod"] == sel.value)
            c["estado"] = "CANCELADA"
            print("Cancelada")

        btn_cons.on_click(consultar)
        btn_can.on_click(cancelar)

        display(sel, btn_cons, btn_can)

# =========================
# HISTORIAL CLÍNICO
# =========================

def vista_historial():
    clear_output()

    if not pacientes:
        print("No hay pacientes")
        return

    doc = widgets.Dropdown(options=[p["doc"] for p in pacientes], description="Paciente:")
    btn = widgets.Button(description="Ver historial")

    def ver(b):
        for c in consultas:
            if c["doc"] == doc.value:
                print(c)

    btn.on_click(ver)

    display(doc, btn)

# =========================
# MENÚ PRINCIPAL
# =========================

menu = widgets.Dropdown(
    options=[
        ("Pacientes", 1),
        ("Médicos", 2),
        ("Citas", 3),
        ("Historial", 4)
    ],
    description="Módulo:"
)

btn_ir = widgets.Button(description="Ir")

def navegar(b):
    if menu.value == 1:
        vista_pacientes()
    elif menu.value == 2:
        vista_medicos()
    elif menu.value == 3:
        vista_citas()
    elif menu.value == 4:
        vista_historial()

btn_ir.on_click(navegar)

display(menu, btn_ir)

Text(value='', description='Documento:')

Text(value='', description='Nombre:')

Text(value='', description='Nacimiento:')

Dropdown(description='Sangre:', options=('O+', 'O-', 'A+', 'A-', 'B+', 'B-', 'AB+', 'AB-'), value='O+')

Text(value='', description='EPS:')

Dropdown(description='Régimen:', options=('CONTRIBUTIVO', 'SUBSIDIADO'), value='CONTRIBUTIVO')

Text(value='', description='Antecedentes:')

Button(description='Guardar', style=ButtonStyle())